# ⚽ Fußball Match Outcome Predictor – Datenpipeline

**Projekt:** KI & Intelligence Engineering  
**Institution:** DHBW Mannheim  

## 📌 Beschreibung

Dieses Skript bildet die Datenpipeline für einen Fußball-Match-Outcome-Predictor.  
Es bereitet historische Spieldaten für das anschließende Modelltraining auf.

Die Pipeline umfasst folgende Schritte:

1. **Datenimport**
   - Lädt historische internationale Fußballergebnisse direkt von GitHub.
   - Datenquelle: [`martj42/international_results`](https://github.com/martj42/international_results)

2. **Elo-Rating-Berechnung**
   - Berechnet dynamische Teamstärken mithilfe des Elo-Systems.
   - Berücksichtigt die historische Performance der Mannschaften.

3. **Feature Engineering**
   - Erstellt relevante Features für Machine-Learning-Modelle.
   - Bereitet die Daten für das Training des Predictors vor.

4. **Datenspeicherung**
   - Speichert den aufbereiteten Datensatz für die weitere Verarbeitung.

---

## 🚀 Verwendung

Das Skript kann direkt über die Kommandozeile ausgeführt werden:

```bash
python datenpipeline.py


**imports**

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
import os

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)

**Konfig**

In [3]:
START_YEAR = 1994
ROLLING_WINDOW = 5  # Formkurve über letzte 5 Spiele
OUTPUT_DIR = "data/processed"

# GitHub-URL für den Datensatz (martj42 – wird regelmäßig aktualisiert)
DATA_URL = "https://raw.githubusercontent.com/martj42/international_results/master/results.csv"

print("=" * 60)
print("⚽  Fußball Match Outcome Predictor – Datenpipeline")
print("=" * 60)
print(f"Startjahr: {START_YEAR} | Rolling Window: {ROLLING_WINDOW} Spiele")
print(f"Datenquelle: {DATA_URL}")

⚽  Fußball Match Outcome Predictor – Datenpipeline
Startjahr: 1994 | Rolling Window: 5 Spiele
Datenquelle: https://raw.githubusercontent.com/martj42/international_results/master/results.csv


# Lade Datensatz von Github

In [4]:
print("" + "─" * 60)
print("📂 Schritt 1: Lade Datensatz von GitHub")
print("─" * 60)

matches = pd.read_csv(DATA_URL, parse_dates=["date"])

print(f"✅ Geladen: {matches.shape[0]:,} Spiele, {matches.shape[1]} Spalten")
print(f"Spalten: {matches.columns.tolist()}")
print(f"Erste Zeilen:")
print(matches.head())

────────────────────────────────────────────────────────────
📂 Schritt 1: Lade Datensatz von GitHub
────────────────────────────────────────────────────────────
✅ Geladen: 49,520 Spiele, 9 Spalten
Spalten: ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral']
Erste Zeilen:
        date home_team away_team  home_score  away_score tournament     city  \
0 1872-11-30  Scotland   England           0           0   Friendly  Glasgow   
1 1873-03-08   England  Scotland           4           2   Friendly   London   
2 1874-03-07  Scotland   England           2           1   Friendly  Glasgow   
3 1875-03-06   England  Scotland           2           2   Friendly   London   
4 1876-03-04  Scotland   England           3           0   Friendly  Glasgow   

    country  neutral  
0  Scotland    False  
1   England    False  
2  Scotland    False  
3   England    False  
4  Scotland    False  


# EDA & Datapreprocessing

In [5]:
print("" + "─" * 60)
print("📊 Schritt 2: Daten-Exploration")
print("─" * 60)

matches['year'] = matches['date'].dt.year

print(f"Zeitraum: {matches['year'].min()} – {matches['year'].max()}")
print(f"Einzigartige Teams: {matches['home_team'].nunique()}")
print(f"Turniere: {matches['tournament'].nunique()}")

print(f"Top 10 Turniere:")
print(matches['tournament'].value_counts().head(10))

# Zielvariable erstellen
matches['target'] = matches.apply(
    lambda row: 0 if row['home_score'] > row['away_score']
    else (1 if row['home_score'] == row['away_score'] else 2),
    axis=1
)

print(f"📈 Ergebnisverteilung:")
print(matches['target'].value_counts(normalize=True).round(3))
print("   0 = Heimsieg | 1 = Unentschieden | 2 = Auswärtssieg")

────────────────────────────────────────────────────────────
📊 Schritt 2: Daten-Exploration
────────────────────────────────────────────────────────────
Zeitraum: 1872 – 2026
Einzigartige Teams: 328
Turniere: 201
Top 10 Turniere:
tournament
Friendly                                18387
FIFA World Cup qualification             8771
UEFA Euro qualification                  2824
African Cup of Nations qualification     2327
FIFA World Cup                           1068
Copa América                              869
African Cup of Nations                    845
AFC Asian Cup qualification               829
UEFA Nations League                       658
CECAFA Cup                                620
Name: count, dtype: int64
📈 Ergebnisverteilung:
target
0    0.490
2    0.283
1    0.227
Name: proportion, dtype: float64
   0 = Heimsieg | 1 = Unentschieden | 2 = Auswärtssieg


# Filtere Daten ab 1994

In [6]:
print("" + "─" * 60)
print("🔍 Schritt 3: Filtere Daten ab 1994")
print("─" * 60)

matches_filtered = matches[matches['year'] >= START_YEAR].copy()
matches_filtered = matches_filtered.sort_values('date').reset_index(drop=True)

print(f"Vorher:  {len(matches):,} Spiele")
print(f"Nachher: {len(matches_filtered):,} Spiele")
print(f"Reduktion: {len(matches) - len(matches_filtered):,} Spiele entfernt ({(1 - len(matches_filtered)/len(matches))*100:.1f}%)")

────────────────────────────────────────────────────────────
🔍 Schritt 3: Filtere Daten ab 1994
────────────────────────────────────────────────────────────
Vorher:  49,520 Spiele
Nachher: 30,017 Spiele
Reduktion: 19,503 Spiele entfernt (39.4%)


# Elo-Ratings berechnen

In [7]:
print("" + "─" * 60)
print("⚡ Schritt 4: Elo-Ratings")
print("─" * 60)

def calculate_elo_ratings(matches_df, initial_rating=1500, k_factor=40):
    """
    Berechnet Elo-Ratings für alle Teams chronologisch.
    Wichtig: Kein Data Leakage – nur vergangene Spiele nutzen!
    """
    ratings = {}
    rating_history = []

    for idx, row in matches_df.iterrows():
        home = row['home_team']
        away = row['away_team']
        date = row['date']

        if home not in ratings:
            ratings[home] = initial_rating
        if away not in ratings:
            ratings[away] = initial_rating

        home_rating = ratings[home]
        away_rating = ratings[away]

        # Erwartungswerte
        E_home = 1 / (1 + 10 ** ((away_rating - home_rating) / 400))
        E_away = 1 / (1 + 10 ** ((home_rating - away_rating) / 400))

        # Tatsächliches Ergebnis
        if row['home_score'] > row['away_score']:
            S_home, S_away = 1, 0
        elif row['home_score'] == row['away_score']:
            S_home, S_away = 0.5, 0.5
        else:
            S_home, S_away = 0, 1

        # Rating-Updates
        ratings[home] += k_factor * (S_home - E_home)
        ratings[away] += k_factor * (S_away - E_away)

        # Speichere historische Ratings (VOR dem Spiel)
        rating_history.append({
            'date': date, 'team': home, 'elo_rating': home_rating,
            'is_home': True, 'opponent': away, 'opponent_elo': away_rating
        })
        rating_history.append({
            'date': date, 'team': away, 'elo_rating': away_rating,
            'is_home': False, 'opponent': home, 'opponent_elo': home_rating
        })

    return pd.DataFrame(rating_history)

print("🧮 Berechne Elo-Ratings...")
elo_history = calculate_elo_ratings(matches_filtered)
print(f"✅ Elo-Ratings berechnet: {len(elo_history):,} Einträge")

# Merge Elo-Ratings in Hauptdatensatz
home_elo = elo_history[elo_history['is_home'] == True][
    ['date', 'team', 'elo_rating', 'opponent_elo']
].rename(columns={
    'team': 'home_team', 'elo_rating': 'home_elo', 'opponent_elo': 'away_elo'
})

matches_merged = matches_filtered.merge(
    home_elo, on=['date', 'home_team'], how='left'
)

print(f"Spiele mit Elo-Daten: {matches_merged['home_elo'].notna().sum():,} / {len(matches_merged):,}")

# Zeige Top-Teams nach Elo
latest_elo = elo_history.groupby('team')['elo_rating'].last().sort_values(ascending=False)
print(f"🏆 Top 10 Teams nach Elo-Rating:")
print(latest_elo.head(10).round(1))

────────────────────────────────────────────────────────────
⚡ Schritt 4: Elo-Ratings
────────────────────────────────────────────────────────────
🧮 Berechne Elo-Ratings...
✅ Elo-Ratings berechnet: 60,034 Einträge
Spiele mit Elo-Daten: 30,027 / 30,027
🏆 Top 10 Teams nach Elo-Rating:
team
Argentina    2117.7
Spain        2112.4
France       2039.0
England      1986.6
Brazil       1984.8
Portugal     1978.4
Morocco      1967.7
Colombia     1963.9
Mexico       1955.2
Germany      1947.6
Name: elo_rating, dtype: float64


# Feature Engineering

In [8]:
print("" + "─" * 60)
print("🔧 Schritt 5: Feature Engineering")
print("─" * 60)

# --- 5.1 Turnierkategorie ---
def categorize_tournament(tournament):
    t = tournament.lower()
    if 'world cup' in t or 'fifa world cup' in t:
        return 'World Cup'
    elif 'euro' in t or 'european championship' in t:
        return 'Euro'
    elif 'qualification' in t or 'qualifier' in t:
        return 'Qualifier'
    elif 'friendly' in t or 'unofficial' in t:
        return 'Friendly'
    elif 'nations league' in t:
        return 'Nations League'
    else:
        return 'Other'

matches_merged['tournament_cat'] = matches_merged['tournament'].apply(categorize_tournament)
print(f"🏆 Turnierkategorien:")
print(matches_merged['tournament_cat'].value_counts())

# --- 5.2 Formkurve (Rolling Statistics) ---
print("📊 Berechne Formkurven...")

team_games = []
for idx, row in matches_merged.iterrows():
    date = row['date']

    if row['home_score'] > row['away_score']:
        home_points, away_points = 3, 0
    elif row['home_score'] == row['away_score']:
        home_points, away_points = 1, 1
    else:
        home_points, away_points = 0, 3

    team_games.append({
        'date': date, 'team': row['home_team'], 'points': home_points, 'is_home': True
    })
    team_games.append({
        'date': date, 'team': row['away_team'], 'points': away_points, 'is_home': False
    })

team_df = pd.DataFrame(team_games).sort_values(['team', 'date'])

# Rolling Mean (Punkte) – WICHTIG: Shift um 1 für No-Leakage!
team_df['form_points'] = team_df.groupby('team')['points'].rolling(
    window=ROLLING_WINDOW, min_periods=1
).mean().reset_index(level=0, drop=True)
team_df['form_points'] = team_df.groupby('team')['form_points'].shift(1)

# Merge zurück
home_form = team_df[team_df['is_home'] == True][['date', 'team', 'form_points']].rename(
    columns={'team': 'home_team', 'form_points': 'home_form'}
)
away_form = team_df[team_df['is_home'] == False][['date', 'team', 'form_points']].rename(
    columns={'team': 'away_team', 'form_points': 'away_form'}
)

matches_merged = matches_merged.merge(home_form, on=['date', 'home_team'], how='left')
matches_merged = matches_merged.merge(away_form, on=['date', 'away_team'], how='left')

# Fehlende Werte (neue Teams ohne Historie)
matches_merged['home_form'] = matches_merged['home_form'].fillna(1.0)
matches_merged['away_form'] = matches_merged['away_form'].fillna(1.0)

print("✅ Formkurven berechnet")

# --- 5.3 Tordifferenz (Rolling) ---
print("⚽ Berechne Tordifferenzen...")

team_goals = []
for idx, row in matches_merged.iterrows():
    date = row['date']
    team_goals.append({
        'date': date, 'team': row['home_team'],
        'goals_scored': row['home_score'], 'goals_conceded': row['away_score'], 'is_home': True
    })
    team_goals.append({
        'date': date, 'team': row['away_team'],
        'goals_scored': row['away_score'], 'goals_conceded': row['home_score'], 'is_home': False
    })

goals_df = pd.DataFrame(team_goals).sort_values(['team', 'date'])

goals_df['avg_goals_scored'] = goals_df.groupby('team')['goals_scored'].rolling(
    window=ROLLING_WINDOW, min_periods=1
).mean().reset_index(level=0, drop=True)
goals_df['avg_goals_conceded'] = goals_df.groupby('team')['goals_conceded'].rolling(
    window=ROLLING_WINDOW, min_periods=1
).mean().reset_index(level=0, drop=True)

# Shift für No-Leakage
goals_df['avg_goals_scored'] = goals_df.groupby('team')['avg_goals_scored'].shift(1)
goals_df['avg_goals_conceded'] = goals_df.groupby('team')['avg_goals_conceded'].shift(1)

# Merge zurück
home_goals = goals_df[goals_df['is_home'] == True][['date', 'team', 'avg_goals_scored', 'avg_goals_conceded']].rename(
    columns={'team': 'home_team', 'avg_goals_scored': 'home_avg_scored', 'avg_goals_conceded': 'home_avg_conceded'}
)
away_goals = goals_df[goals_df['is_home'] == False][['date', 'team', 'avg_goals_scored', 'avg_goals_conceded']].rename(
    columns={'team': 'away_team', 'avg_goals_scored': 'away_avg_scored', 'avg_goals_conceded': 'away_avg_conceded'}
)

matches_merged = matches_merged.merge(home_goals, on=['date', 'home_team'], how='left')
matches_merged = matches_merged.merge(away_goals, on=['date', 'away_team'], how='left')

# Fehlende Werte
for col in ['home_avg_scored', 'home_avg_conceded', 'away_avg_scored', 'away_avg_conceded']:
    matches_merged[col] = matches_merged[col].fillna(1.0)

print("✅ Tordifferenzen berechnet")

# --- 5.4 Head-to-Head ---
print("🤝 Berechne Head-to-Head Bilanzen...")

df = matches_merged.sort_values('date').copy()
df['pair_id'] = df.apply(
    lambda row: '|||'.join(sorted([row['home_team'], row['away_team']])), axis=1
)

h2h_wins, h2h_draws, h2h_losses = [], [], []

for pair_id, group in df.groupby('pair_id'):
    group = group.sort_values('date')
    wins = draws = losses = 0

    for idx, row in group.iterrows():
        h2h_wins.append(wins)
        h2h_draws.append(draws)
        h2h_losses.append(losses)

        if row['home_score'] > row['away_score']:
            wins += 1
        elif row['home_score'] == row['away_score']:
            draws += 1
        else:
            losses += 1

df['h2h_home_wins'] = h2h_wins
df['h2h_draws'] = h2h_draws
df['h2h_home_losses'] = h2h_losses

total_h2h = df['h2h_home_wins'] + df['h2h_draws'] + df['h2h_home_losses']
df['h2h_win_rate'] = np.where(total_h2h > 0, df['h2h_home_wins'] / total_h2h, 0.33)

matches_merged = df.copy()
print("✅ H2H berechnet")

# --- 5.5 Elo-Differenz & Neutral ---
matches_merged['elo_diff'] = matches_merged['home_elo'] - matches_merged['away_elo']
matches_merged['is_neutral'] = matches_merged['neutral'].astype(int)

print("✅ Elo-Differenz & Neutral berechnet")



────────────────────────────────────────────────────────────
🔧 Schritt 5: Feature Engineering
────────────────────────────────────────────────────────────
🏆 Turnierkategorien:
tournament_cat
Friendly          10020
World Cup          7147
Other              6274
Qualifier          3149
Euro               2357
Nations League     1080
Name: count, dtype: int64
📊 Berechne Formkurven...
✅ Formkurven berechnet
⚽ Berechne Tordifferenzen...
✅ Tordifferenzen berechnet
🤝 Berechne Head-to-Head Bilanzen...
✅ H2H berechnet
✅ Elo-Differenz & Neutral berechnet


# Finale Feature-Matrix

In [9]:
print("" + "─" * 60)
print("📋 Schritt 6: Finale Feature-Matrix")
print("─" * 60)

feature_columns = [
    'home_elo', 'away_elo', 'elo_diff',
    'home_form', 'away_form',
    'home_avg_scored', 'home_avg_conceded',
    'away_avg_scored', 'away_avg_conceded',
    'h2h_home_wins', 'h2h_draws', 'h2h_home_losses', 'h2h_win_rate',
    'is_neutral',
]

target_column = 'target'
meta_columns = ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament_cat']

final_df = matches_merged[meta_columns + feature_columns + [target_column]].copy()
final_df = final_df.dropna(subset=feature_columns)

print(f"Shape: {final_df.shape}")
print(f"Features ({len(feature_columns)}):")
for i, col in enumerate(feature_columns, 1):
    print(f"  {i:2d}. {col}")

print(f"Zielvariable: {target_column} (0=Heimsieg, 1=Unentschieden, 2=Auswärtssieg)")
print(f"Klassenverteilung:")
print(final_df[target_column].value_counts(normalize=True).round(3))

────────────────────────────────────────────────────────────
📋 Schritt 6: Finale Feature-Matrix
────────────────────────────────────────────────────────────
Shape: (341149, 21)
Features (14):
   1. home_elo
   2. away_elo
   3. elo_diff
   4. home_form
   5. away_form
   6. home_avg_scored
   7. home_avg_conceded
   8. away_avg_scored
   9. away_avg_conceded
  10. h2h_home_wins
  11. h2h_draws
  12. h2h_home_losses
  13. h2h_win_rate
  14. is_neutral
Zielvariable: target (0=Heimsieg, 1=Unentschieden, 2=Auswärtssieg)
Klassenverteilung:
target
0    0.906
2    0.049
1    0.045
Name: proportion, dtype: float64


# Train/Test-Split nach Zeit

In [10]:
print("" + "─" * 60)
print("📅 Schritt 7: Zeitlicher Train/Test-Split")
print("─" * 60)

TRAIN_END = '2018-01-01'
VAL_END = '2022-01-01'

train_df = final_df[final_df['date'] < TRAIN_END].copy()
val_df = final_df[(final_df['date'] >= TRAIN_END) & (final_df['date'] < VAL_END)].copy()
test_df = final_df[final_df['date'] >= VAL_END].copy()

print(f"Train:   {train_df['date'].min().date()} – {train_df['date'].max().date()} | {len(train_df):,} Spiele")
print(f"Val:     {val_df['date'].min().date()} – {val_df['date'].max().date()} | {len(val_df):,} Spiele")
print(f"Test:    {test_df['date'].min().date()} – {test_df['date'].max().date()} | {len(test_df):,} Spiele")

print("Klassenverteilung:")
for name, df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    dist = df[target_column].value_counts(normalize=True).round(3)
    print(f"  {name}: H={dist.get(0,0):.3f}, D={dist.get(1,0):.3f}, A={dist.get(2,0):.3f}")

────────────────────────────────────────────────────────────
📅 Schritt 7: Zeitlicher Train/Test-Split
────────────────────────────────────────────────────────────
Train:   1994-01-02 – 2017-12-29 | 316,517 Spiele
Val:     2018-01-02 – 2021-12-31 | 3,540 Spiele
Test:    2022-01-01 – 2026-07-19 | 21,092 Spiele
Klassenverteilung:
  Train: H=0.939, D=0.016, A=0.045
  Val: H=0.478, D=0.232, A=0.290
  Test: H=0.495, D=0.439, A=0.066


# Speichern

In [11]:
print("" + "─" * 60)
print("💾 Schritt 8: Speichern")
print("─" * 60)

os.makedirs(OUTPUT_DIR, exist_ok=True)

final_df.to_csv(f"{OUTPUT_DIR}/matches_full.csv", index=False)
train_df.to_csv(f"{OUTPUT_DIR}/train.csv", index=False)
val_df.to_csv(f"{OUTPUT_DIR}/val.csv", index=False)
test_df.to_csv(f"{OUTPUT_DIR}/test.csv", index=False)
pd.DataFrame({'feature': feature_columns}).to_csv(f"{OUTPUT_DIR}/features.csv", index=False)

print(f"✅ Daten gespeichert in {OUTPUT_DIR}/:")
print(f"  matches_full.csv  ({len(final_df):,} Zeilen)")
print(f"  train.csv       ({len(train_df):,} Zeilen)")
print(f"  val.csv         ({len(val_df):,} Zeilen)")
print(f"  test.csv        ({len(test_df):,} Zeilen)")
print(f"  features.csv    ({len(feature_columns)} Features)")

────────────────────────────────────────────────────────────
💾 Schritt 8: Speichern
────────────────────────────────────────────────────────────
✅ Daten gespeichert in data/processed/:
  matches_full.csv  (341,149 Zeilen)
  train.csv       (316,517 Zeilen)
  val.csv         (3,540 Zeilen)
  test.csv        (21,092 Zeilen)
  features.csv    (14 Features)


# Zusammenfassung

In [12]:
print("" + "=" * 60)
print("✅  Datenpipeline abgeschlossen!")
print("=" * 60)
print("""
Nächste Schritte:
  1. Modelltraining:  python modelltraining.py
  2. Streamlit-App:   streamlit run app.py
""")

✅  Datenpipeline abgeschlossen!

Nächste Schritte:
  1. Modelltraining:  python modelltraining.py
  2. Streamlit-App:   streamlit run app.py

